In [ ]:
!pip install transformers torch gradio accelerate bitsandbytes sentencepiece reportlab -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 46.0 MB/s eta 0:00:00


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import gradio as gr
import re
from datetime import datetime
import json
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Preformatted
from reportlab.lib.enums import TA_LEFT
import warnings
warnings.filterwarnings('ignore')

print("🔄 Loading IBM Granite 3.3 2B Instruct model...")
print("⏳ This may take 3-5 minutes on first run...")

model_name = "ibm-granite/granite-3.0-2b-instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

print("✅ Model loaded successfully!")
print("✅ Tokenizer ready!")

🔄 Loading IBM Granite 3.3 2B Instruct model...
⏳ This may take 3-5 minutes on first run...


config.json:   0%|          | 0.00/785 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Model loaded successfully!
✅ Tokenizer ready!


In [3]:
DATABASE_SCHEMA = """
**📊 Database Schema:**

**1. users table:**
- id (INTEGER, PRIMARY KEY)
- name (TEXT)
- email (TEXT)
- age (INTEGER)
- country (TEXT)
- created_at (DATE)

**2. orders table:**
- id (INTEGER, PRIMARY KEY)
- user_id (INTEGER, FOREIGN KEY)
- product_id (INTEGER, FOREIGN KEY)
- order_date (DATE)
- amount (DECIMAL)
- status (TEXT)

**3. products table:**
- id (INTEGER, PRIMARY KEY)
- product_name (TEXT)
- category (TEXT)
- price (DECIMAL)
- stock (INTEGER)

**4. transactions table:**
- id (INTEGER, PRIMARY KEY)
- order_id (INTEGER, FOREIGN KEY)
- transaction_date (DATETIME)
- payment_method (TEXT)
- amount (DECIMAL)
"""

# Simple schema text for model prompt
SCHEMA_FOR_MODEL = """
Database Tables:
1. users (id, name, email, age, country, created_at)
2. orders (id, user_id, product_id, order_date, amount, status)
3. products (id, product_name, category, price, stock)
4. transactions (id, order_id, transaction_date, payment_method, amount)
"""

# Conversation history storage
conversation_history = []

print("✅ Database schema loaded!")
print("✅ Storage initialized!")

✅ Database schema loaded!
✅ Storage initialized!


In [4]:
def generate_sql_from_text(user_input):
    """Generate SQL query from natural language using AI model"""
    try:
        # Create prompt for the model
        prompt = f"""You are an expert SQL query generator. Convert the natural language request into a valid SQL query.

{SCHEMA_FOR_MODEL}

Natural Language Request: {user_input}

Generate ONLY the SQL query without any explanation. The query should be:
- Syntactically correct
- Safe (no SQL injection)
- Optimized for performance
- Use proper JOINs when multiple tables are needed

SQL Query:"""

        # Prepare input for model
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            return_dict=True
        ).to(model.device)

        # Generate SQL query
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

        # Decode the response
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract and clean SQL
        sql_query = extract_and_clean_sql(response)

        return sql_query

    except Exception as e:
        return f"-- Error generating query: {str(e)}"

def extract_and_clean_sql(response):
    """Extract SQL query from model response and clean it"""
    # Remove everything before the SQL query starts
    if "SQL Query:" in response:
        response = response.split("SQL Query:")[-1]

    # Look for SQL keywords
    sql_pattern = r'(SELECT|INSERT|UPDATE|DELETE|CREATE|WITH)[\s\S]*?(?:;|$)'
    match = re.search(sql_pattern, response, re.IGNORECASE | re.MULTILINE)

    if match:
        sql = match.group(0).strip()
        # Remove markdown code blocks if present
        sql = re.sub(r'```sql\s*|\s*```', '', sql)
        # Add semicolon if missing
        if not sql.endswith(';'):
            sql += ';'
        return sql

    # Fallback: look for lines with SQL keywords
    lines = response.split('\n')
    for i, line in enumerate(lines):
        if any(kw in line.upper() for kw in ['SELECT', 'INSERT', 'UPDATE', 'DELETE']):
            # Get this line and potentially next lines
            sql_lines = [line]
            for next_line in lines[i+1:]:
                if next_line.strip():
                    sql_lines.append(next_line)
                if ';' in next_line:
                    break
            sql = ' '.join(sql_lines).strip()
            if not sql.endswith(';'):
                sql += ';'
            return sql

    # Last resort
    response = response.strip()
    if not response.endswith(';'):
        response += ';'
    return response

print("✅ SQL generation functions loaded!")

✅ SQL generation functions loaded!


In [5]:
def export_to_pdf():
    """Export all conversation history to PDF file"""
    if not conversation_history:
        return None, "⚠️ No queries to export. Generate some queries first!"

    try:
        # Create filename with timestamp
        filename = f"sql_queries_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"

        # Create PDF document
        doc = SimpleDocTemplate(filename, pagesize=letter, topMargin=0.5*inch)
        story = []
        styles = getSampleStyleSheet()

        # Custom styles
        title_style = ParagraphStyle(
            'CustomTitle',
            parent=styles['Heading1'],
            fontSize=24,
            textColor='#1f77b4',
            spaceAfter=30,
            alignment=TA_LEFT
        )

        subtitle_style = ParagraphStyle(
            'Subtitle',
            parent=styles['Normal'],
            fontSize=12,
            textColor='#666666',
            spaceAfter=20
        )

        query_title_style = ParagraphStyle(
            'QueryTitle',
            parent=styles['Heading2'],
            fontSize=14,
            textColor='#2c3e50',
            spaceAfter=10
        )

        code_style = ParagraphStyle(
            'Code',
            parent=styles['Code'],
            fontSize=9,
            leftIndent=20,
            backColor='#f5f5f5',
            borderColor='#ddd',
            borderWidth=1,
            borderPadding=5
        )

        # Add title
        story.append(Paragraph("🤖 SQL Query Generator - Export", title_style))

        # Add metadata
        info_text = f"""
        <b>Generated:</b> {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}<br/>
        <b>Total Queries:</b> {len(conversation_history)}<br/>
        <b>Model:</b> IBM Granite 3.3 2B Instruct
        """
        story.append(Paragraph(info_text, subtitle_style))
        story.append(Spacer(1, 0.3*inch))

        # Add each query
        for idx, item in enumerate(conversation_history, 1):
            # Query header
            query_header = f"<b>Query #{idx}</b> &nbsp;&nbsp; <i>{item['timestamp']}</i>"
            story.append(Paragraph(query_header, query_title_style))
            story.append(Spacer(1, 0.1*inch))

            # Natural language input
            input_label = "<b>Natural Language Request:</b>"
            story.append(Paragraph(input_label, styles['Normal']))
            input_text = item['input'].replace('<', '&lt;').replace('>', '&gt;')
            story.append(Paragraph(input_text, styles['Normal']))
            story.append(Spacer(1, 0.1*inch))

            # Generated SQL
            sql_label = "<b>Generated SQL Query:</b>"
            story.append(Paragraph(sql_label, styles['Normal']))
            sql_text = item['output'].replace('<', '&lt;').replace('>', '&gt;')
            story.append(Preformatted(sql_text, code_style))
            story.append(Spacer(1, 0.3*inch))

            # Add separator line except for last item
            if idx < len(conversation_history):
                story.append(Spacer(1, 0.1*inch))

        # Build PDF
        doc.build(story)

        return filename, f"✅ Successfully exported {len(conversation_history)} queries to PDF!"

    except Exception as e:
        return None, f"❌ Error creating PDF: {str(e)}"

def export_to_txt():
    """Export all conversation history to TXT file"""
    if not conversation_history:
        return None, "⚠️ No queries to export. Generate some queries first!"

    try:
        # Create filename with timestamp
        filename = f"sql_queries_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

        with open(filename, 'w', encoding='utf-8') as f:
            # Write header
            f.write("=" * 80 + "\n")
            f.write("SQL QUERY GENERATOR - EXPORT FILE\n")
            f.write("=" * 80 + "\n\n")

            # Write metadata
            f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total Queries: {len(conversation_history)}\n")
            f.write(f"Model: IBM Granite 3.3 2B Instruct\n")
            f.write("\n" + "=" * 80 + "\n\n")

            # Write each query
            for idx, item in enumerate(conversation_history, 1):
                f.write(f"Query #{idx}\n")
                f.write(f"Timestamp: {item['timestamp']}\n")
                f.write("\n")

                f.write("Natural Language Request:\n")
                f.write(f"{item['input']}\n")
                f.write("\n")

                f.write("Generated SQL Query:\n")
                f.write(f"{item['output']}\n")
                f.write("\n")
                f.write("-" * 80 + "\n\n")

        return filename, f"✅ Successfully exported {len(conversation_history)} queries to TXT!"

    except Exception as e:
        return None, f"❌ Error creating TXT file: {str(e)}"

def clear_conversation():
    """Clear all conversation history"""
    global conversation_history
    conversation_history = []
    return [], "✅ History cleared successfully!"

print("✅ Export functions loaded (PDF & TXT)!")

✅ Export functions loaded (PDF & TXT)!


In [6]:
def chat_interface(message, history):
    """Handle chat-style conversation"""
    if not message or not message.strip():
        return history, ""

    try:
        # Generate SQL query
        sql_output = generate_sql_from_text(message.strip())

        # Determine which tables were likely used
        tables_used = []
        if 'users' in message.lower() or 'user' in message.lower() or 'customer' in message.lower():
            tables_used.append('users')
        if 'order' in message.lower():
            tables_used.append('orders')
        if 'product' in message.lower():
            tables_used.append('products')
        if 'transaction' in message.lower() or 'payment' in message.lower():
            tables_used.append('transactions')

        # If no specific table mentioned, analyze SQL
        if not tables_used:
            sql_lower = sql_output.lower()
            if 'users' in sql_lower:
                tables_used.append('users')
            if 'orders' in sql_lower:
                tables_used.append('orders')
            if 'products' in sql_lower:
                tables_used.append('products')
            if 'transactions' in sql_lower:
                tables_used.append('transactions')

        tables_info = f"*Generated from table(s): {', '.join(tables_used)}*" if tables_used else "*Query generated successfully*"

        # Format response
        response = f"**✅ SQL GENERATED**\n\n```sql\n{sql_output}\n```\n\n{tables_info}"

        # Add to conversation history
        conversation_history.append({
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "input": message.strip(),
            "output": sql_output
        })

        # Update chat history
        history.append((message, response))

        return history, ""

    except Exception as e:
        error_msg = f"❌ Error: {str(e)}"
        history.append((message, error_msg))
        return history, ""

print("✅ Chat interface function loaded!")


✅ Chat interface function loaded!


In [7]:
custom_css = """
.gradio-container {
    background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%) !important;
}
.contain {
    background: rgba(30, 60, 114, 0.95) !important;
}
#chatbot {
    background: #1a1a2e !important;
    border-radius: 10px !important;
    border: 2px solid #3a506b !important;
}
.message-wrap {
    background: #16213e !important;
}
.user-message {
    background: #0e4c92 !important;
}
.bot-message {
    background: #1c2833 !important;
}
"""

# Create Gradio interface
with gr.Blocks(title="SQL Query Generator", theme=gr.themes.Soft(), css=custom_css) as app:

    # Header
    gr.Markdown("""
    # 🤖 SQL Query Generator
    ## Natural Language → SQLite SQL
    ### Powered by IBM Granite 3.3 2B Instruct

    Convert plain English requests into safe, production-ready SQL queries!
    """)

    with gr.Row():
        # Left side - Chat area
        with gr.Column(scale=3):
            # Chatbot
            chatbot = gr.Chatbot(
                label="💬 Query Conversation",
                height=500,
                show_copy_button=True,
                elem_id="chatbot",
                avatar_images=(None, "🤖")
            )

            # Input area
            with gr.Row():
                msg = gr.Textbox(
                    placeholder="Example: Show me all the users who signed up for last 2 days",
                    label="📝 Natural Language Request",
                    lines=2,
                    scale=4,
                    autofocus=True
                )

                generate_btn = gr.Button(
                    "🚀 Generate SQL",
                    variant="primary",
                    scale=1,
                    size="lg"
                )

            # Example queries
            gr.Examples(
                examples=[
                    "Show me all the users who signed up for last 2 days",
                    "Find total revenue from each product category",
                    "List all pending orders with customer details",
                    "Get top 5 products by sales amount",
                    "Show customers from USA who are older than 25",
                    "What is the average order amount per customer?",
                    "Find users who haven't placed any orders yet",
                    "Show products with stock less than 10 units"
                ],
                inputs=msg,
                label="💡 Click any example to try"
            )

        # Right side - Controls and Schema
        with gr.Column(scale=1):
            # Controls section
            gr.Markdown("## 🎛️ Controls")

            clear_btn = gr.Button(
                "🗑️ Clear History",
                variant="secondary",
                size="sm"
            )

            clear_status = gr.Textbox(
                label="Status",
                interactive=False,
                lines=1,
                visible=False
            )

            gr.Markdown("---")

            # Export section
            gr.Markdown("## 📤 Export")

            pdf_btn = gr.Button(
                "📄 Download PDF",
                variant="primary",
                size="sm"
            )

            txt_btn = gr.Button(
                "📄 Download TXT",
                variant="primary",
                size="sm"
            )

            export_status = gr.Textbox(
                label="Export Status",
                interactive=False,
                lines=2
            )

            pdf_file = gr.File(
                label="📥 PDF File",
                visible=False
            )

            txt_file = gr.File(
                label="📥 TXT File",
                visible=False
            )

            gr.Markdown("---")

            # Database schema
            gr.Markdown(DATABASE_SCHEMA)

    # Event handlers

    # Generate SQL on button click
    def handle_generate(message, history):
        new_history, cleared_msg = chat_interface(message, history)
        return new_history, cleared_msg

    generate_btn.click(
        fn=handle_generate,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg]
    )

    # Generate SQL on Enter key
    msg.submit(
        fn=handle_generate,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg]
    )

    # Clear history
    def handle_clear():
        cleared_chat, status = clear_conversation()
        return cleared_chat, status

    clear_btn.click(
        fn=handle_clear,
        inputs=None,
        outputs=[chatbot, export_status]
    )

    # Export to PDF
    def handle_pdf_export():
        file, status = export_to_pdf()
        return file, status, gr.File(visible=True) if file else gr.File(visible=False)

    pdf_btn.click(
        fn=handle_pdf_export,
        inputs=None,
        outputs=[pdf_file, export_status, pdf_file]
    )

    # Export to TXT
    def handle_txt_export():
        file, status = export_to_txt()
        return file, status, gr.File(visible=True) if file else gr.File(visible=False)

    txt_btn.click(
        fn=handle_txt_export,
        inputs=None,
        outputs=[txt_file, export_status, txt_file]
    )

print("✅ Gradio interface created!")


✅ Gradio interface created!


In [ ]:
print("\n" + "=" * 70)
print("🚀 Launching SQL Query Generator...")
print("=" * 70)
print("⏳ Please wait while the interface starts...")
print()

# Launch the app
app.launch(
    share=True,
    debug=True,
    show_error=True
)

print("\n✅ Application launched successfully!")
print("📱 Click the URL above to open your SQL Query Generator!")
print("💡 Tip: The URL will work for 72 hours")



🚀 Launching SQL Query Generator...
⏳ Please wait while the interface starts...

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3464c982dd1fdf5b33.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
